In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ahora importa el módulo
from RRAM.Representate import config_ax

def setup_plt(plt, latex=True, scaling=1):
    plt.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": latex,
            "font.family": "fourier",
            "text.latex.preamble": "\n".join(
                [
                    r"\usepackage[utf8]{inputenc}",
                    r"\usepackage[T1]{fontenc}",
                    r"\usepackage{siunitx}",
                    r"\usepackage{physics}",
                ]
            ),
        }
    )

    SMALL_SIZE = 8 * scaling
    MEDIUM_SIZE = 10 * scaling
    BIGGER_SIZE = 11 * scaling

    plt.rc("font", size=SMALL_SIZE)
    plt.rc("axes", titlesize=SMALL_SIZE)
    plt.rc("axes", labelsize=MEDIUM_SIZE)
    plt.rc("xtick", labelsize=SMALL_SIZE)
    plt.rc("ytick", labelsize=SMALL_SIZE)
    plt.rc("legend", fontsize=SMALL_SIZE)
    plt.rc("figure", titlesize=BIGGER_SIZE)
    plt.rc("axes", titlesize=BIGGER_SIZE * 1.05)

In [ ]:
# Rutas de los archivos
ruta_raiz = Path.cwd()
ruta_datos = (
    ruta_raiz
    / "Datos_Experimentales"
    / "Medidas_Experimentales_RRAM"
    
)

ruta_figuras = ruta_raiz / "Datos_Experimentales" / "V_Set"

archivos = [archivo for archivo in ruta_datos.glob("Cycle_p_1659*.txt")]

print(f"Archivos encontrados: {len(archivos)}")

In [ ]:
def plot_iv_and_derivative(V_set, I_set, dI_dV, top_indices, ruta_figuras):
    """
    Plots the I-V curve and its derivative, highlighting specific points with the highest derivative values.
    Parameters:
    -----------
    V_set : array-like
        Array of voltage values (x-axis data).
    I_set : array-like
        Array of current values corresponding to the voltage values (y-axis data for the I-V curve).
    dI_dV : array-like
        Array of derivative values (y-axis data for the derivative plot).
    top_indices : list of int
        Indices of the points with the highest derivative values to be highlighted on the plots.
    ruta_figuras : str
        File path where the generated figure will be saved.
    Returns:
    --------
    None
        The function saves the generated plot as an image file at the specified location.
    Notes:
    ------
    - The function creates a figure with two subplots:
        1. The I-V curve with logarithmic scaling on the y-axis.
        2. The derivative of the I-V curve.
    - Points with the highest derivative values are highlighted in red on both subplots.
    - The figure is saved as an image file in the specified path with a resolution of 300 dpi.
    """
    # Crear una figura con dos subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # === Configuración de estilo LaTeX y plot ===
    setup_plt(plt, latex=True, scaling=2)

    config_ax(ax1)
    config_ax(ax2)

    # En el primer subplot, graficar la curva I-V
    ax1.plot(V_set, I_set, "b-", linewidth=2)
    ax1.set_xlabel("Voltaje (V)")
    ax1.set_ylabel("Corriente (A)")
    ax1.set_title("Curva I-V")
    ax1.set_yscale("log")
    ax1.grid(True)

    # Solo marcar los puntos con mayor derivada en el primer subplot
    for idx in top_indices:
        ax1.plot(V_set[idx], I_set[idx], "go", markersize=6, color="red")
        ax1.annotate(
            f"{V_set[idx]:.2f} V",
            (V_set[idx], I_set[idx]),
            textcoords="offset points",
            xytext=(-35, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    # En el segundo subplot, graficar la derivada de la corriente respecto al voltaje
    ax2.plot(V_set, dI_dV, "g-", linewidth=2, color = "blue")
    ax2.set_xlabel("Voltaje (V)")
    ax2.set_ylabel("dI/dV (A/V)")
    ax2.set_ylabel(r"$\dv{I}{V}$ (\si{\ampere/\volt})")
    ax2.set_title("Derivada de la curva I-V")
    ax2.grid(True)

    # Solo marcar los puntos con mayor derivada en el segundo subplot
    for idx in top_indices:
        ax2.plot(V_set[idx], dI_dV[idx], "go", markersize=6, color="red")
        ax2.annotate(
            f"{V_set[idx]:.2f} V",
            (V_set[idx], dI_dV[idx]),
            textcoords="offset points",
            xytext=(-25, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )
   
    fig.savefig(ruta_figuras, dpi=300, bbox_inches="tight")  # type: ignore
    plt.close(fig)
    

In [ ]:
def obtener_V_set(ruta_datos, num_valores_maximos=3):
    """
    Analiza un archivo de datos experimentales para calcular la derivada de la corriente 
    con respecto al voltaje, identifica los valores máximos de la derivada y genera una 
    gráfica de los datos.
    Args:
        ruta_datos (Path): Ruta al archivo que contiene los datos experimentales. 
            El archivo debe existir y contener tres columnas: "Voltaje", "Corriente" y "tiempo".
        num_valores_maximos (int, opcional): Número de valores máximos de la derivada a extraer. 
            Por defecto es 5.
    Returns:
        dict: Un diccionario con las siguientes claves:
            - "top_derivadas" (np.ndarray): Los valores máximos de la derivada numérica.
            - "top_voltajes" (np.ndarray): Los voltajes correspondientes a los valores máximos 
              de la derivada.
    Raises:
        FileNotFoundError: Si el archivo especificado en `ruta_datos` no existe.
    Notas:
        - La función filtra los datos para considerar solo las primeras 111 filas del archivo.
        - Genera una gráfica que muestra la curva IV y la derivada, y la guarda en una 
          subcarpeta "Datos_Experimentales/V_Set" con un nombre basado en el archivo de entrada.
    """
    # Verificar que el archivo existe
    if not ruta_datos.exists():
        raise FileNotFoundError(
            f"El archivo {ruta_datos} no existe. Verificar la ruta."
        )

    # Cargar datos experimentales
    data_set = np.loadtxt(ruta_datos)

    # Crear el DataFrame
    df_set = pd.DataFrame(data_set, columns=["Voltaje", "Corriente", "tiempo"])
    df_set_filtrado = df_set.iloc[0:110]  # Seleccionar solo las primeras 111 filas

    # Calcular la derivada numérica de la corriente con respecto al voltaje
    dI_dV = np.gradient(df_set_filtrado['Corriente'], df_set_filtrado['Voltaje'])

    # Extraer los índices de los valores máximos
    indices_maximos = np.argsort(dI_dV)[-num_valores_maximos:]

    # Extraer los valores máximos y sus voltajes correspondientes
    top_derivadas = dI_dV[indices_maximos]
    top_voltajes = df_set_filtrado["Voltaje"].values[indices_maximos]
    
    V_set = df_set_filtrado["Voltaje"]
    I_set = df_set_filtrado["Corriente"]
    
    ruta_figuras = (
        Path.cwd()
        / "Datos_Experimentales"
        / "V_Set"
        / f"{ruta_datos.stem}_IV_derivada.png"
    )
    plot_iv_and_derivative(V_set, I_set, dI_dV, indices_maximos, ruta_figuras)
    
    # Retornar resultados en un diccionario
    return {
        "top_derivadas": top_derivadas,
        "top_voltajes": top_voltajes,
    }

In [ ]:
raiz_figuras = (
    Path.cwd()
    / "Datos_Experimentales"
    / "V_Set"
    / f"{archivos[0].stem}_IV_derivada.png"
)

print(f"Guardando figuras en: {raiz_figuras}")


for datos in archivos:
    print(f"Archivo: {datos.stem}")
    
    resultados = obtener_V_set(datos, num_valores_maximos=3)
    
    # Encuentra el índice del voltaje más bajo
    idx_min_voltaje = np.argmin(resultados["top_voltajes"])
    voltaje_min = resultados["top_voltajes"][idx_min_voltaje]
    derivada_min = resultados["top_derivadas"][idx_min_voltaje]

    # Guardo los resultados en un único archivo de texto
    with open(ruta_figuras / "V_set_resumen.txt", "a") as f:
        f.write(f"{datos.stem}\t{voltaje_min:.4f}\t{derivada_min:.4e}\n")

    # for voltaje, derivada in zip(resultados["top_voltajes"], resultados["top_derivadas"]):
    #     print(f"Voltaje: {voltaje:.4f} V, Derivada: {derivada:.4e} A/V")
